# Indian Ocean zarr (`IO.zarr`): EDA

Daily, 0.25° data cube for the Arabian Sea and Bay of Bengal (32°N to 12°S, 42 to 102°E, 1979–2022), from the
[mind-the-chl-gap](https://geo-smart.github.io/mind-the-chl-gap/) project. It holds chlorophyll-a (GlobColour L3 and gap-free L4),
ERA5 winds and temperatures, OSCAR currents, CMEMS sea level, and GLORYS mixed-layer depth and salinity.

* Data (local disk, not s3fs): `~/mind_the_chl_gap/data/IO.zarr` (24.7 GB on disk, **66 GB in memory**; the machine has 15 GB RAM) and the cached summaries `~/mind_the_chl_gap/data/summaries.zarr`
* Source: `gcs://nmfs_odp_nwfsc/CB/mind_the_chl_gap/IO.zarr` (public, anonymous)
* Code (next to this notebook, `~/shared/ecology/mind_the_chl_gap/scripts/`): `io_summaries.py` (one-pass cached summaries), `io_viewer.py` (interactive day viewer)

### Working with it: xarray + dask, not numpy

The project notebooks stay lazy until the end (`xr.open_zarr`, then `.sel`, reduce, and `.compute()` on the small result).
The same rules apply here:

| Do | Avoid | Why |
|---|---|---|
| `xr.open_zarr(path)` (default `chunks={}` → dask, on-disk chunks) | `xr.open_zarr(path, chunks=None)` or `.load()` on the whole cube | `chunks=None` means **no dask**: the first operation loads the full variable (`np.log10` on one variable → 5.7 GB peak here). 66 GB won't fit in 15 GB RAM |
| `.sel()` / `.isel()` **before** computing | `.to_numpy()` / `.values` on a full variable | each 3-D variable is 2.7 GB in memory |
| Reduce lazily (`.mean`, `.groupby`, `.resample`, `np.log(da)`), then `.compute()` | converting to numpy and looping | numpy ufuncs on a DataArray **stay lazy** |
| `dask.compute(a, b, c)` for several results | `a.compute(); b.compute(); c.compute()` | each chunk is read once, not three times |
| Match the chunk layout: chunks are **(100 days × full map)** | pixel time series over all years | one day = 1 chunk per variable; one pixel over 44 years = all 161 chunks |
| Cache expensive reductions to a small zarr (`io_summaries.py`) | recomputing climatologies each session | a full pass takes ~9 min; the cache opens instantly |
| For full-cube reductions, loop over chunk-aligned time blocks and accumulate | one giant graph (`ds.groupby('time.month').mean()` on everything) | bounds peak memory (~1–2 GB) and gives progress output |

**Data notes (from the project docs and checks below)**
* Every variable spans the full 1979–2022 time axis but only has data for its source's years. Outside those years it is NaN.
  CHL starts Sept 1997; OSCAR currents, SLA/ADT and GLORYS start 1993; ERA5 has the full period.
* `CHL_cmes-land`: 0 = ocean, 2 = land. 43 "ocean" pixels are above sea level (Tibetan lakes, SST around 0 °C),
  so the summaries define ocean as `land != 2 & topo <= 0`.
* **ERA5 is missing whole months:** 1,013 days with no SST and 1,209 with no wind, in blocks of about 29 days,
  spread over 1979–1992 and 2021–22 (§5). 1993–2020 is complete.
* **`mlotst` before 1993 is a copy of `u_wind`** (identical in all 181M pixel-days, 1979–1992). The build notebook
  ([IO_mixed_layer_thickness.ipynb](https://github.com/safs-varanasi-internship/indian-ocean-zarr/blob/main/notebooks/working/IO_mixed_layer_thickness.ipynb))
  copied `u_wind` as a template and set `.values[:] = np.nan`. On a dask-backed array that assignment modifies a temporary numpy copy,
  so the NaN fill never happened. Use `mlotst` only for 1993–2020 (2021–22 is NaN).
* `CHL_cmes_flags-gapfree` holds non-integer values (329 unique on 2020-06-15), probably from regridding. Treat it as approximate.

In [ ]:
import time
import numpy as np
import pandas as pd
import xarray as xr
import dask
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm, Normalize

import io_summaries as S
from io_viewer import DayViewer, OKABE_ITO, SEQ, SEQ2, DIV

%matplotlib inline
plt.rcParams.update({"figure.dpi": 90, "axes.grid": False})
xr.set_options(display_expand_data_vars=True, display_expand_attrs=False)

## 1. Open lazily
This reads only metadata. Nothing is loaded until `.compute()`, `.load()`, `.values` or `.to_numpy()`.

In [ ]:
ds = S.open_io()
ds

In [ ]:
# In-memory size vs. chunking per variable (nbytes = uncompressed size if fully loaded)
info = pd.DataFrame(
    {
        v: dict(
            dims=",".join(ds[v].dims),
            dtype=str(ds[v].dtype),
            GB_in_memory=round(ds[v].nbytes / 1e9, 2),
            chunk=str(ds[v].data.chunksize),
            n_chunks=ds[v].data.npartitions,
            units=ds[v].attrs.get("units", ""),
        )
        for v in ds.data_vars
    }
).T
print(
    f"total in memory if loaded: {ds.nbytes/1e9:.0f} GB   |   time steps: {ds.sizes['time']}   grid: {ds.sizes['lat']}×{ds.sizes['lon']}"
)
info

## 2. Lazy vs. eager, and chunk-aware access (quick demo)
The timings below show the table above in practice. The last cell reads a single pixel across 44 years. That touches all 161 chunks of the variable (2.7 GB decompressed) to return 64 kB. It's quick here because of the local disk and OS cache; from the GCS bucket it would be 161 requests.

In [ ]:
chl = ds["CHL_cmes-gapfree"]
log_chl = np.log10(chl)  # numpy ufunc on a DataArray -> still lazy, nothing read
print(
    type(log_chl.data), "| in-memory size if computed:", f"{log_chl.nbytes/1e9:.1f} GB"
)
assert isinstance(
    log_chl.data, dask.array.Array
), "not dask-backed: check open_zarr chunks"

t = time.time()
one_day = chl.sel(time="2020-06-15").compute()
print(
    f"one day (1 chunk):            {time.time()-t:5.2f} s, {one_day.nbytes/1e6:.1f} MB"
)

t = time.time()
year_mean = chl.sel(time="2020").mean(["lat", "lon"]).compute()
print(
    f"2020 domain-mean series (4 chunks, reduced before compute): {time.time()-t:5.2f} s, {year_mean.nbytes/1e3:.1f} kB"
)

# several results from the same chunks: one compute call -> chunks read once
t = time.time()
sst_mean, sst_max = dask.compute(
    ds.sst.sel(time="2020").mean(), ds.sst.sel(time="2020").max()
)
print(
    f"mean+max SST 2020 in one dask.compute: {time.time()-t:5.2f} s -> {float(sst_mean)-273.15:.2f} / {float(sst_max)-273.15:.2f} °C"
)

In [ ]:
# Anti-pattern for this chunk layout: one pixel, all years (reads + decompresses all 161 chunks of sst)
t = time.time()
px = ds.sst.sel(lat=15, lon=65, method="nearest").compute()
print(
    f"single-pixel SST, 1979-2022: {time.time()-t:5.2f} s for {px.nbytes/1e3:.0f} kB of output"
)
# If you need many pixel time series, rechunk once (e.g. chunks={'time': -1, 'lat': 20, 'lon': 20})
# and write a copy to zarr instead of paying this per pixel.

## 3. Daily map viewer
Scroll with the slider, ◀ ▶ buttons, **play**, or **jump to** a date. `step` sets the jump size (day, week, month, year).
The viewer holds one 100-day block of the six panel variables in memory, so steps inside a block are instant.
Crossing into a new block takes about 1–2 s.

*Anomaly* subtracts the monthly climatology from §4 (a log10 ratio for CHL) and needs the cached summaries.
Wind and current arrows show **direction only** (unit length); the colour shows speed. Grey = land or no data. Black line = land-mask coastline.

In [ ]:
daily, clim = (
    (xr.open_zarr(S.OUT, group=g).load() for g in ("daily", "clim"))
    if S.OUT.exists()
    else (None, None)
)
viewer = DayViewer(ds, clim=clim, daily=daily)
viewer.show(start="2020-06-15")

## 4. Cached summaries (single pass over the cube)
`io_summaries.py` loops over 200-day blocks (2 chunks each), with one `dask.compute` per block, and computes:
* per day and variable: ocean-pixel mean, fraction of ocean pixels with data, and an all-zero-map flag
* per calendar month: climatology maps of the main variables, plus the fraction of days each pixel is observed in CHL L3 (i.e., not cloud-covered)

The first run takes about 9 min (4 cores, 2.4 GB peak RAM). After that it loads instantly from `~/mind_the_chl_gap/data/summaries.zarr`.

In [ ]:
if not S.OUT.exists():
    S.main()
daily = xr.open_zarr(S.OUT, group="daily").load()  # small: 16071 days × ~80 series
clim = xr.open_zarr(S.OUT, group="clim").load()  # small: 12 months × 177 × 241
viewer.daily, viewer.clim = (
    daily,
    clim,
)  # enables anomaly mode + time series in the viewer above
vars3d = [v.split("__")[0] for v in daily.data_vars if v.endswith("__mean")]

## 5. Data availability
Fraction of ocean pixels with data, averaged per year. White = no data.

In [ ]:
valid_yr = (
    pd.DataFrame({v: daily[f"{v}__valid"].to_series() for v in vars3d})
    .resample("YE")
    .mean()
)
fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(
    valid_yr.T.values, aspect="auto", cmap=SEQ, vmin=0, vmax=1, interpolation="nearest"
)
ax.set_xticks(range(0, len(valid_yr), 2), valid_yr.index.year[::2], rotation=90)
ax.set_yticks(range(len(vars3d)), vars3d)
ax.set_xlabel("year")
ax.set_ylabel("variable")
ax.set_title(
    "Yearly mean fraction of ocean pixels with data, per variable (n = 16071 days)"
)
plt.colorbar(im, ax=ax, label="fraction of ocean pixels with data")
plt.show()

In [ ]:
rows = {}
for v in vars3d:
    has = daily[f"{v}__valid"].to_series() > 0
    rows[v] = dict(
        first_day=has.idxmax().date() if has.any() else None,
        last_day=has[::-1].idxmax().date() if has.any() else None,
        median_valid_frac=(
            round(daily[f"{v}__valid"].to_series()[has].median(), 3)
            if has.any()
            else np.nan
        ),
        all_zero_days=int(daily[f"{v}__allzero"].sum()),
        days_with_data=int(has.sum()),
    )
avail = pd.DataFrame(rows).T
avail

### CHL L3 coverage (cloud gaps)
Daily fraction of ocean pixels observed in the gappy L3 product, 1997–2022.

In [ ]:
l3 = daily["CHL_cmes-level3__valid"].sel(time=slice(S.CHL_START, None)).to_series()
fig, axs = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw=dict(width_ratios=[3, 1]))
axs[0].plot(l3.index, l3, lw=0.4, color=OKABE_ITO[4], label="daily")
axs[0].plot(
    l3.rolling(90, center=True).mean(),
    color=OKABE_ITO[5],
    lw=1.5,
    label="90-day rolling mean",
)
axs[0].set(
    title=f"CHL L3: daily fraction of ocean pixels observed (n = {len(l3)} days)",
    xlabel="date",
    ylabel="fraction of ocean pixels observed",
)
axs[0].legend()
bym = l3.groupby(l3.index.month)
axs[1].boxplot([g.values for _, g in bym], showfliers=False)
axs[1].set(
    title="CHL L3 observed fraction by calendar month",
    xlabel="month",
    ylabel="fraction of ocean pixels observed",
)
plt.tight_layout()
plt.show()

## 6. Ocean-mean time series
Monthly means of the daily ocean-pixel means. Each panel starts when its data starts.

In [ ]:
ts_vars = [
    ("CHL_cmes-gapfree", "CHL L4 gap-free (mg m$^{-3}$)", 0),
    ("sst", "SST (°C)", -273.15),
    ("air_temp", "2 m air temp (°C)", -273.15),
    ("wind_speed", "10 m wind speed (m s$^{-1}$)", 0),
    ("curr_speed", "current speed (m s$^{-1}$)", 0),
    ("sla", "sea level anomaly (m)", 0),
    ("mlotst", "mixed layer depth (m)", 0),
    ("so", "salinity (PSU)", 0),
]
fig, axs = plt.subplots(len(ts_vars), 1, figsize=(14, 2.1 * len(ts_vars)), sharex=True)
LINES = [c for c in OKABE_ITO if c != "#F0E442"]  # drop yellow: low contrast on white
for ax, (v, lab, off), c in zip(axs, ts_vars, LINES * 2):
    s = (daily[f"{v}__mean"].to_series() + off).resample("MS").mean()
    ax.plot(s.index, s, color=c, lw=1)
    ax.set_ylabel(lab, fontsize=8)
    ax.set_title(
        f"{lab}: ocean-pixel monthly mean (n months = {s.notna().sum()})", fontsize=9
    )
axs[-1].set_xlabel("date")
plt.tight_layout()
plt.show()

## 7. Seasonal cycle: monthly climatology maps
Each variable is averaged over its own available years (CHL 1997–2022, currents and SLA 1993–2022, ERA5 1979–2022).

In [ ]:
land = ds["CHL_cmes-land"].load() == 2
months = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]


def clim_grid(da, title, cbar_label, quiver=None, **kw):
    g = da.plot(
        col="month",
        col_wrap=4,
        figsize=(16, 10),
        add_colorbar=True,
        cbar_kwargs=dict(label=cbar_label, shrink=0.6),
        **kw
    )
    for m, ax in zip(da.month.values, g.axs.flat):
        ax.contour(
            land.lon,
            land.lat,
            land.astype(float),
            levels=[0.5],
            colors="k",
            linewidths=0.5,
        )
        ax.set_facecolor("0.85")
        ax.set_title(months[m - 1])
        ax.set_aspect("equal")
        if quiver:
            st = 10
            u, v = (
                clim[c]
                .sel(month=m)
                .isel(lat=slice(None, None, st), lon=slice(None, None, st))
                for c in quiver
            )
            ax.quiver(
                u.lon,
                u.lat,
                u,
                v,
                color="white",
                scale=120 if quiver[0] == "u_wind" else 8,
                width=0.004,
            )
    g.fig.suptitle(title, y=1.02, fontsize=13)
    plt.show()


clim_grid(
    clim["CHL_cmes-gapfree"],
    "Monthly climatology: CHL L4 gap-free (1997–2022)",
    "CHL (mg m$^{-3}$), log scale",
    cmap=SEQ,
    norm=LogNorm(0.03, 10),
)

In [ ]:
clim_grid(
    clim["CHL_l3_obs_frac"],
    "Monthly climatology: fraction of days observed in CHL L3 (1997–2022)",
    "fraction of days observed",
    cmap=SEQ2,
    vmin=0,
    vmax=1,
)

In [ ]:
clim_grid(
    clim["sst"] - 273.15,
    "Monthly climatology: SST, ERA5 (1979–2022)",
    "SST (°C)",
    cmap="inferno",
    vmin=22,
    vmax=31,
)

In [ ]:
clim_grid(
    clim["wind_speed"],
    "Monthly climatology: 10 m wind speed (colour) and mean wind vector (arrows), ERA5 (1979–2022)",
    "wind speed (m s$^{-1}$)",
    quiver=("u_wind", "v_wind"),
    cmap=SEQ2,
    vmin=0,
    vmax=12,
)

In [ ]:
clim_grid(
    clim["curr_speed"],
    "Monthly climatology: surface current speed (colour) and mean current vector (arrows), OSCAR (1993–2022)",
    "current speed (m s$^{-1}$)",
    quiver=("u_curr", "v_curr"),
    cmap=SEQ2,
    vmin=0,
    vmax=0.8,
)

In [ ]:
clim_grid(
    clim["mlotst"],
    "Monthly climatology: mixed layer depth, GLORYS (1993–2022)",
    "mixed layer depth (m)",
    cmap=SEQ,
    vmin=0,
    vmax=80,
)

## 8. Hovmöller: latitude × time for one year
Longitude-band mean per latitude and day. Pick a region and a year. One year reads 4–5 chunks per variable.

In [ ]:
import ipywidgets as W

REGIONS = {
    "Arabian Sea (55–75°E)": (55, 75),
    "Bay of Bengal (80–95°E)": (80, 95),
    "Full domain (42–102°E)": (42, 102),
}
HOV = [
    (
        "CHL_cmes-gapfree",
        "CHL L4 gap-free (mg m$^{-3}$)",
        dict(cmap=SEQ, norm=LogNorm(0.03, 10)),
        0,
    ),
    (
        "CHL_cmes-level3",
        "CHL L3 observed (mg m$^{-3}$)",
        dict(cmap=SEQ, norm=LogNorm(0.03, 10)),
        0,
    ),
    ("sst", "SST (°C)", dict(cmap="inferno", vmin=22, vmax=31), -273.15),
    ("wind_speed", "10 m wind speed (m s$^{-1}$)", dict(cmap=SEQ2, vmin=0, vmax=14), 0),
]
ocean = S.ocean_mask(ds)


def hovmoller(year, region):
    lo, hi = REGIONS[region]
    sub = (
        ds[[v for v, *_ in HOV]]
        .sel(time=str(year), lon=slice(lo, hi))
        .where(ocean.sel(lon=slice(lo, hi)))
    )
    band = sub.mean("lon").compute()  # reduce lazily, compute the small result once
    fig, axs = plt.subplots(len(HOV), 1, figsize=(15, 3 * len(HOV)), sharex=True)
    for ax, (v, lab, kw, off) in zip(axs, HOV):
        im = ax.pcolormesh(band.time, band.lat, (band[v] + off).T, shading="auto", **kw)
        plt.colorbar(im, ax=ax, label=lab, pad=0.01)
        ax.set_ylabel("latitude (°N)")
        ax.set_facecolor("0.85")
        ax.set_title(f"{lab}, mean over {region}, {year}", fontsize=10)
    axs[-1].set_xlabel("date")
    plt.tight_layout()
    plt.show()


W.interact(
    hovmoller,
    year=W.IntSlider(value=2020, min=1998, max=2022, continuous_update=False),
    region=W.Dropdown(options=list(REGIONS)),
)